# SOLikeT grid yaml generator

This notebook assembles a Cobaya grid yaml (`grid.yaml`) that can be passed directly to:
```
cobaya-grid-create <grid_folder> grid.yaml
```

## File layout expected

```
defaults/
    likelihoods/
        lensing.yaml
        mflike.yaml
        multigaussian_mflike_lensing.yaml
    parameters/
        default_cosmo.yaml
        mflike_fg.yaml
        mflike_sys.yaml
        mnu.yaml
        neff.yaml
        probe_cosmo/
            cosmo_mflike.yaml
            cosmo_lensing.yaml
            cosmo_multigaussian_mflike_lensing.yaml
    sampling/
        sampler.yaml
    theory/
        theory_BandpowerForeground.yaml
        default_camb.yaml
        default_class.yaml
        probe_camb/
            camb_mflike.yaml
            camb_lensing.yaml
            camb_multigaussian_mflike_lensing.yaml
        probe_class/
            class_mflike.yaml
            class_lensing.yaml
            class_multigaussian_mflike_lensing.yaml
```

## Groups and datasets produced

| Group | Models | Datasets |
|---|---|---|
| `lcdm` | `lcdm` | `mflike`, `lensing`, `multigaussian_mflike_lensing` |
| `extended` | `mnu`, `neff`, `mnu_neff` | `mflike`, `lensing`, `multigaussian_mflike_lensing` |

## 1 — Configuration

In [ ]:
import os
from pathlib import Path
from cobaya.model import get_model
from cobaya.tools import resolve_packages_path


SIMS = Path("sims")
DEFAULTS_DIR = Path("defaults")
XCOV = SIMS / "XCov_mflike_lensing.fits"
OUTPUT_YAML   = Path("grid.yaml")
OUTPUT_DIR    = Path("output_grid")

# --- Models to include
LCDM_MODELS     = ["lcdm"]
EXTENDED_MODELS = ["mnu", "neff", "mnu_neff"]

# --- Theory code to use ("camb" or "class")
THEORY_CODE = "camb"

## 2 — Load defaults and model yamls

In [ ]:
import yaml

def load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)

# Load default parameters common to all chains
defaults = {
    "sampler": load_yaml(DEFAULTS_DIR / "sampling/sampler.yaml"),
    "params": load_yaml(DEFAULTS_DIR / "parameters/default_cosmo.yaml"),
    "theory": load_yaml(DEFAULTS_DIR / f"theory/default_{THEORY_CODE}.yaml"),
}

# Load parameters which are specific to a given theoretical model
all_model_names = LCDM_MODELS + EXTENDED_MODELS
model_params = {}
for name in all_model_names:
    parts = name.split("_")
    if len(parts) > 1 and all((DEFAULTS_DIR / f"parameters/{p}.yaml").exists() for p in parts):
        combined = {}
        for p in parts:
            combined.update(load_yaml(DEFAULTS_DIR / f"parameters/{p}.yaml")["params"])
        model_params[name] = combined
    elif (DEFAULTS_DIR / f"parameters/{parts[0]}.yaml").exists():
        model_params[name] = load_yaml(DEFAULTS_DIR / f"parameters/{name}.yaml")["params"]
    else:
        model_params[name] = {}

print("Models loaded:   ", all_model_names)

## 3 — Build dataset info dicts
We load likelihoods and parameter/theory yaml files specific to each likelihood

In [ ]:
# -- mflike only
info_mflike = load_yaml(DEFAULTS_DIR / "likelihoods/mflike.yaml") \
                | load_yaml(DEFAULTS_DIR / "parameters/probe_cosmo/cosmo_mflike.yaml")
                
info_mflike["params"] = load_yaml(DEFAULTS_DIR / "parameters/mflike_fg.yaml") \
                        | load_yaml(DEFAULTS_DIR / "parameters/mflike_sys.yaml")

info_mflike["theory"] = load_yaml(DEFAULTS_DIR / "theory/theory_BandpowerForeground.yaml") \
                        | load_yaml(DEFAULTS_DIR / f"theory/probe_{THEORY_CODE}/{THEORY_CODE}_mflike.yaml")

# -- lensing only
info_lensing = load_yaml(DEFAULTS_DIR / "likelihoods/lensing.yaml") \
                | load_yaml(DEFAULTS_DIR / "parameters/probe_cosmo/cosmo_lensing.yaml")
            
info_lensing["theory"] = load_yaml(DEFAULTS_DIR / f"theory/probe_{THEORY_CODE}/{THEORY_CODE}_lensing.yaml")

# -- joint multigaussian ( including xcov)
# THIS WILL BE UPDATED TO SIMPLY READ likelihoods/multigaussian_mflike_lensinglike.yaml
likelihood_names = [list(info_mflike["likelihood"].keys())[0],
                    list(info_lensing["likelihood"].keys())[0]]

options = [info_mflike["likelihood"][likelihood_names[0]],
           info_lensing["likelihood"][likelihood_names[1]]]

info_mflike_lensing = {
    "likelihood": {
        "soliket.gaussian.MultiGaussianLikelihood": {
            "components": likelihood_names,
            "options": options,
            "cross_cov_path": str(XCOV.resolve()),
            "stop_at_error": True
        }
    },
    "params": load_yaml(DEFAULTS_DIR / "parameters/mflike_fg.yaml") \
            | load_yaml(DEFAULTS_DIR / "parameters/mflike_sys.yaml") \
            | load_yaml(DEFAULTS_DIR / "parameters/probe_cosmo/cosmo_multigaussian_mflike_lensing.yaml")["params"],
    "theory": load_yaml(DEFAULTS_DIR / "theory/theory_BandpowerForeground.yaml") \
                | load_yaml(DEFAULTS_DIR / f"theory/probe_{THEORY_CODE}/{THEORY_CODE}_multigaussian_mflike_lensing.yaml")
}

datasets = {
    "mflike":             info_mflike,
    "lensing":            info_lensing,
    "multigaussian_mflike_lensing": info_mflike_lensing,
}

## 4 — Assemble and write `grid.yaml`

In [ ]:
from cobaya.tools import resolve_packages_path

PACKAGES_STR = str(resolve_packages_path())

grid = {
    # ---------------------------------------------------------------- defaults
    "defaults": defaults,

    # ------------------------------------------------------------------ groups
    "groups": {
        "lcdm": {
            "models":   LCDM_MODELS,
            "datasets": list(datasets.keys()),
            "base": THEORY_CODE,
        },
        "extended": {
            "models":   EXTENDED_MODELS,
            "datasets": list(datasets.keys()),
            "base": THEORY_CODE,
        },
    },

    # ------------------------------------------------------------------ models
    
    "models": {
        name: {"params": params}
        for name, params in model_params.items()
    },

    # ----------------------------------------------------------------- datasets
    "datasets": datasets,

    # ----------------------------------------------------------------- covmats
    "cov_dir": None,  # set once first runs are available
}

# Write yaml grid file
header = (
    "# SOLikeT grid yaml -- generated by generate_grid.ipynb.\n"
    "# Do not edit by hand; re-run the notebook to regenerate.\n"
    "#   cobaya-grid-create <grid_folder> grid.yaml\n\n"
)
import json
grid_clean = json.loads(json.dumps(grid))
text = yaml.dump(grid_clean, sort_keys=False, default_flow_style=False, allow_unicode=True)

OUTPUT_YAML.write_text(header + text)
print(f"Written: {OUTPUT_YAML}  ({OUTPUT_YAML.stat().st_size} bytes)")
print()
print("Next step:")
print(f"  cobaya-grid-create <grid_folder> {OUTPUT_YAML}")



## 5 — Generate grid

In [ ]:
from cobaya.grid_tools.gridconfig import grid_create
grid_create((str(OUTPUT_DIR.resolve()), str(OUTPUT_YAML.resolve())))